# Pré-processamento de dados

> **Autor**: Miguel Vieira Machado Pim  
> **Contexto**: Desafio do processo seletivo de estágio IndustriALL

Este notebook irá abordar toda a parte de pré-processamento de dados, unindo todos os datasets para obtermos um único dataset que possa ser utilizado nas próximas etapas.

## Bibliotecas

In [1]:
import pandas as pd
from pathlib import Path

## Carregando os dados

In [2]:
data_path = Path("../data")

feature_dfs = []

for file in data_path.glob("*.csv"):
    if "target" in file.name:
        target_df = pd.read_csv(file)
    else:
        feature_dfs.append(pd.read_csv(file))

In [3]:
feature_dfs[0].head()

,timestamp,TAG_iALL_PS_37
0,2018-04-01 00:00:00,156.908398
1,2018-04-01 00:01:00,160.845276
2,2018-04-01 00:02:00,157.841402
3,2018-04-01 00:03:00,156.106507
4,2018-04-01 00:04:00,163.462727


In [4]:
target_df.head()

,timestamp,target_iALL_PS
0,2018-04-01 00:00:00,NORMAL
1,2018-04-01 00:01:00,NORMAL
2,2018-04-01 00:02:00,NORMAL
3,2018-04-01 00:03:00,NORMAL
4,2018-04-01 00:04:00,NORMAL


## Tratando nome das colunas

#### Feature dataframes

In [5]:
for i, df in enumerate(feature_dfs):
    columns = ["timestamp", f"sensor_{i}"]
    
    df.columns = columns

In [6]:
feature_dfs[0].head()

,timestamp,sensor_0
0,2018-04-01 00:00:00,156.908398
1,2018-04-01 00:01:00,160.845276
2,2018-04-01 00:02:00,157.841402
3,2018-04-01 00:03:00,156.106507
4,2018-04-01 00:04:00,163.462727


#### Target dataframe

In [7]:
columns = ["timestamp", "target"]
target_df.columns = columns

target_df.head()

,timestamp,target
0,2018-04-01 00:00:00,NORMAL
1,2018-04-01 00:01:00,NORMAL
2,2018-04-01 00:02:00,NORMAL
3,2018-04-01 00:03:00,NORMAL
4,2018-04-01 00:04:00,NORMAL


## Tratando os tipos de cada dataframe

#### Feature dataframes

In [8]:
for i, df in enumerate(feature_dfs):
    print(f"DataFrame {i}")
    print(df.dtypes)
    print()

DataFrame 0
timestamp     object
sensor_0     float64
dtype: object

DataFrame 1
timestamp     object
sensor_1     float64
dtype: object

DataFrame 2
timestamp     object
sensor_2     float64
dtype: object

DataFrame 3
timestamp     object
sensor_3     float64
dtype: object

DataFrame 4
timestamp     object
sensor_4     float64
dtype: object

DataFrame 5
timestamp     object
sensor_5     float64
dtype: object

DataFrame 6
timestamp     object
sensor_6     float64
dtype: object

DataFrame 7
timestamp     object
sensor_7     float64
dtype: object

DataFrame 8
timestamp     object
sensor_8     float64
dtype: object

DataFrame 9
timestamp     object
sensor_9     float64
dtype: object

DataFrame 10
timestamp     object
sensor_10    float64
dtype: object

DataFrame 11
timestamp     object
sensor_11    float64
dtype: object

DataFrame 12
timestamp     object
sensor_12    float64
dtype: object

DataFrame 13
timestamp     object
sensor_13    float64
dtype: object

DataFrame 14
timestamp     obj

Todos os datasets tem as suas colunas numéricas com o tipo correto, portanto vamos ajustar apenas a coluna do tipo data.

In [9]:
for df in feature_dfs:
    df["timestamp"] = pd.to_datetime(df["timestamp"])

In [10]:
feature_dfs[0].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  220320 non-null  datetime64[ns]
 1   sensor_0   220304 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 3.4 MB


#### Target dataframe

In [11]:
target_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   timestamp  220320 non-null  object
 1   target     220320 non-null  object
dtypes: object(2)
memory usage: 3.4+ MB


In [ ]:
# Mapeando as classes para valores numéricos
target_df["target"] = target_df["target"].map({
    "NORMAL": 0,
    "ANORMAL": 1
}).astype(int)

# Transformando a coluna de timestamp para o tipo datetime
target_df["timestamp"] = pd.to_datetime(target_df["timestamp"])

In [13]:
target_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  220320 non-null  datetime64[ns]
 1   target     220320 non-null  int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 3.4 MB


## Unindo os datasets

Com todos os datasets tratados, o objetivo agora é unificar eles em um único dataset para ser utilizado posteriormente.

In [14]:
df_final = (
    pd.concat(
        [target_df.set_index("timestamp")] +
        [df.set_index("timestamp") for df in feature_dfs],
        axis=1,
        join="outer" # Para mantermos timestamps que não estão em todas as tabelas
    )
    .reset_index()
)

df_final.head()

,timestamp,target,sensor_0,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,...,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51
0,2018-04-01 00:00:00,0,156.908398,310.022461,101.547023,71.130001,9.271792,1513.429451,197.014054,33.066858,...,426.651658,140.957957,223.192894,69.511954,4.548754,1842.687291,721.402437,120.531554,2121.942413,1463.647280
1,2018-04-01 00:01:00,0,160.845276,306.084796,71.846296,98.697513,7.989979,1264.045436,549.491578,165.013373,...,375.316113,148.975866,1225.833795,92.131989,7.887998,1810.309041,656.074488,94.319273,1914.209147,1437.274642
2,2018-04-01 00:02:00,0,157.841402,353.863854,59.174616,39.329786,8.771413,1357.757041,710.303108,68.023284,...,444.809188,94.262093,42.455318,86.164997,4.975919,1508.671751,1060.796664,40.182165,1805.064984,1565.113515
3,2018-04-01 00:03:00,0,156.106507,301.563110,0.674927,126.751846,9.234399,1416.592337,451.379204,140.740890,...,414.052496,154.828040,525.719895,32.473642,6.304142,1483.740243,719.418238,64.405477,1295.933558,1966.270851
4,2018-04-01 00:04:00,0,163.462727,298.957820,71.785623,41.526627,9.101509,1468.721792,1014.853751,113.162924,...,431.548430,137.733814,950.064772,37.866648,1.671733,1600.106478,995.108386,31.432756,1765.669551,1396.339609


In [15]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 54 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  220320 non-null  datetime64[ns]
 1   target     220320 non-null  int64         
 2   sensor_0   220304 non-null  float64       
 3   sensor_1   220293 non-null  float64       
 4   sensor_2   220293 non-null  float64       
 5   sensor_3   220301 non-null  float64       
 6   sensor_4   220301 non-null  float64       
 7   sensor_5   220248 non-null  float64       
 8   sensor_6   220304 non-null  float64       
 9   sensor_7   220293 non-null  float64       
 10  sensor_8   220304 non-null  float64       
 11  sensor_9   220293 non-null  float64       
 12  sensor_10  215213 non-null  float64       
 13  sensor_11  220304 non-null  float64       
 14  sensor_12  220293 non-null  float64       
 15  sensor_13  220301 non-null  float64       
 16  sensor_14  215522 no

## Conclusão

Definindo uma função para executar todos os passos vistos neste notebook. Essa função será utilizada nos próximos notebooks.

In [16]:
def preprocess_data(data_path: Path) -> pd.DataFrame:
    """
    Essa função coleta todos os 53 arquivos csv e converte eles em um único dataframe final.

    Args:
        data_path (Path): Caminho para a pasta que contém os arquivos csv.

    Returns:
        pd.DataFrame: Dataframe final contendo todas as features e a coluna target.
    """
    # Coletando todos os arquivos
    feature_dfs = []

    for file in data_path.glob("*.csv"):
        if "target" in file.name:
            target_df = pd.read_csv(file)
        else:
            feature_dfs.append(pd.read_csv(file))
    
    # Renomeando as colunas
    for i, df in enumerate(feature_dfs):
        columns = ["timestamp", f"sensor_{i}"]
        
        df.columns = columns
    
    columns = ["timestamp", "target"]
    target_df.columns = columns
    
    # Tratando tipos dos datasets
    for df in feature_dfs:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    target_df["target"] = target_df["target"].map({
        "NORMAL": 0,
        "ANORMAL": 1
    }).astype(int)
    target_df["timestamp"] = pd.to_datetime(target_df["timestamp"])
    
    # Construindo dataset final
    df_final = (
        pd.concat(
            [target_df.set_index("timestamp")] +
            [df.set_index("timestamp") for df in feature_dfs],
            axis=1,
            join="outer" # Para mantermos timestamps que não estão em todas as tabelas
        )
        .reset_index()
    )
    
    return df_final